In [1]:
#Loading synthetic data with real world data

import pandas as pd

train_df = pd.read_csv(r"C:\Users\shiro\OneDrive\Desktop\Python files\VIz Project\data\Maternal Health Risk Data Set.csv")

test_df = pd.read_csv(r"C:\Users\shiro\OneDrive\Desktop\Python files\VIz Project\data\synthetic_maternal_data_cleaned.csv")

In [2]:
# Standardize RiskLevel labels in both datasets
train_df["RiskLevel"] = train_df["RiskLevel"].str.replace("_", " ").str.lower().str.strip()
test_df["RiskLevel"] = test_df["RiskLevel"].str.replace("_", " ").str.lower().str.strip()


In [3]:
#Feature Engineering on training data

# Feature Engineering on training data
train_df["MAP"] = (train_df["SystolicBP"] + 2 * train_df["DiastolicBP"]) / 3
train_df["PulsePressure"] = train_df["SystolicBP"] - train_df["DiastolicBP"]
train_df["ShockIndex"] = train_df["HeartRate"] / train_df["SystolicBP"]
train_df["BPRatio"] = train_df["SystolicBP"] / train_df["DiastolicBP"]

train_df["TempDeviation"] = train_df["BodyTemp"] - 98.2
train_df["BSDeviation"] = train_df["BS"] - 7

train_df["CombinedRiskScore"] = (
    (train_df["MAP"] > 105).astype(int) +
    (train_df["BS"] > 10).astype(int) +
    (train_df["HeartRate"] > 90).astype(int) +
    (train_df["TempDeviation"] > 1).astype(int)
)


In [4]:
#Feature engineering on testing data

# Feature Engineering on test data
test_df["MAP"] = (test_df["SystolicBP"] + 2 * test_df["DiastolicBP"]) / 3
test_df["PulsePressure"] = test_df["SystolicBP"] - test_df["DiastolicBP"]
test_df["ShockIndex"] = test_df["HeartRate"] / test_df["SystolicBP"]
test_df["BPRatio"] = test_df["SystolicBP"] / test_df["DiastolicBP"]

test_df["TempDeviation"] = test_df["BodyTemp"] - 98.2
test_df["BSDeviation"] = test_df["BS"] - 7

test_df["CombinedRiskScore"] = (
    (test_df["MAP"] > 105).astype(int) +
    (test_df["BS"] > 10).astype(int) +
    (test_df["HeartRate"] > 90).astype(int) +
    (test_df["TempDeviation"] > 1).astype(int)
)


In [5]:
#Preparing X and y
X_train = train_df.drop("RiskLevel" , axis = 1)
y_train = train_df["RiskLevel"]

In [6]:
#Real(Evaluation)
X_test = test_df.drop("RiskLevel", axis=1)
y_test = test_df["RiskLevel"]

In [7]:
#Encode target and Scale features

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
#Training single models: 
#1. Logistic Regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter = 2000, multi_class="multinomial")
lr.fit(X_train_scaled, y_train_enc)

c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'multinomial'


In [9]:
#2. Decision Tree

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train_enc)

,criterion,'gini'
,splitter,'best'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [10]:
#3 Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)
rf.fit(X_train, y_train_enc)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
#4. Gradient Boosting

from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train_enc)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [12]:
#Voting Ensemble to average the predicted prob of each model

from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ('lr',lr),
        ('rf', rf),
        ('gb',gb)
    ],
    voting='soft'
)

voting.fit(X_train_scaled, y_train_enc)

c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,estimators,"[('lr', ...), ('rf', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [13]:
#Stacking Ensemble

from sklearn.ensemble import StackingClassifier

stack = StackingClassifier(
    estimators=[
        ('lr', lr),
        ('rf',rf),
        ('gb',gb)
    ],
    final_estimator=LogisticRegression(max_iter=2000),
    passthrough=False
)
stack.fit(X_train_scaled, y_train_enc)

c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_log

,estimators,"[('lr', ...), ('rf', ...), ...]"
,final_estimator,LogisticRegre...max_iter=2000)
,cv,None
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [14]:
#MODEL EVAL

from sklearn.metrics import recall_score, classification_report

def evaluate_recall(model, X_test, y_test, name):
    print(f"\n===== {name} — Recall =====")
    y_pred = model.predict(X_test)

    # Macro recall = average recall across all classes
    macro_recall = recall_score(y_test, y_pred, average='macro')

    # Weighted recall = weighted by class frequency
    weighted_recall = recall_score(y_test, y_pred, average='weighted')

    print("Macro Recall:", macro_recall)
    print("Weighted Recall:", weighted_recall)
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    return macro_recall, weighted_recall


In [15]:
rec_lr, wrec_lr = evaluate_recall(lr, X_test_scaled, y_test_enc, "Logistic Regression")
rec_dt, wrec_dt = evaluate_recall(dt, X_test, y_test_enc, "Decision Tree")
rec_rf, wrec_rf = evaluate_recall(rf, X_test, y_test_enc, "Random Forest")
rec_gb, wrec_gb = evaluate_recall(gb, X_test, y_test_enc, "Gradient Boosting")
rec_vote, wrec_vote = evaluate_recall(voting, X_test_scaled, y_test_enc, "Voting Ensemble")
rec_stack, wrec_stack = evaluate_recall(stack, X_test_scaled, y_test_enc, "Stacking Ensemble")



===== Logistic Regression — Recall =====
Macro Recall: 0.5927879667682158
Weighted Recall: 0.45266666666666666
              precision    recall  f1-score   support

   high risk       0.04      1.00      0.09        36
    low risk       0.95      0.52      0.67      2142
    mid risk       0.21      0.26      0.23       822

    accuracy                           0.45      3000
   macro avg       0.40      0.59      0.33      3000
weighted avg       0.73      0.45      0.54      3000


===== Decision Tree — Recall =====
Macro Recall: 0.5434588271642801
Weighted Recall: 0.3953333333333333
              precision    recall  f1-score   support

   high risk       0.03      0.92      0.06        36
    low risk       0.94      0.43      0.59      2142
    mid risk       0.27      0.28      0.28       822

    accuracy                           0.40      3000
   macro avg       0.41      0.54      0.31      3000
weighted avg       0.74      0.40      0.50      3000


===== Random Forest 

In [16]:
# Voting Ensemble Recall
rec_vote, wrec_vote = evaluate_recall(
    voting, 
    X_test_scaled, 
    y_test_enc, 
    "Voting Ensemble"
)

# Stacking Ensemble Recall
rec_stack, wrec_stack = evaluate_recall(
    stack, 
    X_test_scaled, 
    y_test_enc, 
    "Stacking Ensemble"
)



===== Voting Ensemble — Recall =====
Macro Recall: 0.5790260142986635
Weighted Recall: 0.455
              precision    recall  f1-score   support

   high risk       0.03      0.97      0.07        36
    low risk       0.93      0.53      0.68      2142
    mid risk       0.25      0.23      0.24       822

    accuracy                           0.46      3000
   macro avg       0.41      0.58      0.33      3000
weighted avg       0.74      0.46      0.55      3000


===== Stacking Ensemble — Recall =====
Macro Recall: 0.5829596234276354
Weighted Recall: 0.5526666666666666
              precision    recall  f1-score   support

   high risk       0.03      0.86      0.06        36
    low risk       0.92      0.68      0.78      2142
    mid risk       0.45      0.21      0.29       822

    accuracy                           0.55      3000
   macro avg       0.47      0.58      0.38      3000
weighted avg       0.78      0.55      0.64      3000



In [17]:
recall_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "Voting Ensemble",
        "Stacking Ensemble"
    ],
    "Macro Recall": [
        rec_lr,
        rec_dt,
        rec_rf,
        rec_gb,
        rec_vote,
        rec_stack
    ],
    "Weighted Recall": [
        wrec_lr,
        wrec_dt,
        wrec_rf,
        wrec_gb,
        wrec_vote,
        wrec_stack
    ]
})

recall_results.sort_values(by="Macro Recall", ascending=False)


,Model,Macro Recall,Weighted Recall
0,Logistic Regression,0.592788,0.452667
5,Stacking Ensemble,0.582960,0.552667
4,Voting Ensemble,0.579026,0.455000
1,Decision Tree,0.543459,0.395333
2,Random Forest,0.534019,0.446000
3,Gradient Boosting,0.506740,0.458000


In [18]:
#Trying XGB and Light GBM to handle imbalanced data
#Training XGBoost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss'
)

xgb.fit(X_train_scaled, y_train_enc)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [20]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    objective='multiclass',
    num_class=3
)

lgb.fit(X_train_scaled, y_train_enc)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000435 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 310
[LightGBM] [Info] Number of data points in the train set: 1014, number of used features: 13
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [21]:
rec_xgb, wrec_xgb = evaluate_recall(xgb, X_test_scaled, y_test_enc, "XGBoost")
rec_lgb, wrec_lgb = evaluate_recall(lgb, X_test_scaled, y_test_enc, "LightGBM")



===== XGBoost — Recall =====
Macro Recall: 0.5342853280809485
Weighted Recall: 0.4260000000000001
              precision    recall  f1-score   support

   high risk       0.04      0.92      0.07        36
    low risk       0.87      0.52      0.65      2142
    mid risk       0.18      0.17      0.17       822

    accuracy                           0.43      3000
   macro avg       0.36      0.53      0.30      3000
weighted avg       0.67      0.43      0.51      3000


===== LightGBM — Recall =====
Macro Recall: 0.528042441631965
Weighted Recall: 0.42333333333333334
              precision    recall  f1-score   support

   high risk       0.03      0.92      0.06        36
    low risk       0.87      0.52      0.65      2142
    mid risk       0.18      0.15      0.16       822

    accuracy                           0.42      3000
   macro avg       0.36      0.53      0.29      3000
weighted avg       0.67      0.42      0.51      3000



c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [22]:
#Adding to comparison table
recall_results.loc[len(recall_results)] = ["XGBoost", rec_xgb, wrec_xgb]
recall_results.loc[len(recall_results)] = ["LightGBM", rec_lgb, wrec_lgb]

recall_results.sort_values(by="Macro Recall", ascending=False)


,Model,Macro Recall,Weighted Recall
0,Logistic Regression,0.592788,0.452667
5,Stacking Ensemble,0.582960,0.552667
4,Voting Ensemble,0.579026,0.455000
1,Decision Tree,0.543459,0.395333
6,XGBoost,0.534285,0.426000
2,Random Forest,0.534019,0.446000
7,LightGBM,0.528042,0.423333
3,Gradient Boosting,0.506740,0.458000
